## Étape 2 — Contrôle qualité des données

Vérifie la cohérence du portefeuille : colonnes présentes, dates cohérentes,
valeurs valides pour `cause_sortie` et `sexe`. Les lignes non conformes sont supprimées.

In [ ]:
n_init = len(data)
required_cols = ['id','date_naissance','date_entree','date_sortie','cause_sortie','sexe']
missing = set(required_cols) - set(data.columns)
if missing:
    raise ValueError('Colonnes manquantes : ' + str(missing))
for col in ['date_naissance','date_entree','date_sortie']:
    data[col] = pd.to_datetime(data[col], errors='coerce')
mask = (
    (data['date_entree'] < data['date_sortie']) &
    (data['date_naissance'] < data['date_entree']) &
    (data['date_entree'] >= pd.Timestamp('1900-01-01')) &
    (data['cause_sortie'].isin(['deces','autre'])) &
    (data['sexe'].isin(['H','F'])) &
    (data[required_cols].notna().all(axis=1))
)
data = data[mask].copy()
data = data[data['sexe'] == SEXE].copy()
n_suppr = n_init - len(data)
data['age_entree'] = (data['date_entree'] - data['date_naissance']).dt.days / 365.25
data['age_sortie'] = (data['date_sortie'] - data['date_naissance']).dt.days / 365.25
data['duree_obs_ans'] = (data['date_sortie'] - data['date_entree']).dt.days / 365.25
print('Lignes initiales       : {:,}'.format(n_init))
print('Lignes supprimees (QC) : {:,}'.format(n_suppr))
print('Lignes conservees      : {:,}'.format(len(data)))
print('Sexe analyse : ' + SEXE)
print()
print('Ages a l entree :')
print(data['age_entree'].describe().round(1).to_string())
print()
print('Duree observation (annees) :')
print(data['duree_obs_ans'].describe().round(2).to_string())
print()
print('Causes de sortie :')
print(data['cause_sortie'].value_counts().to_string())
n_age_hors = ((data['age_entree'] < 20) | (data['age_entree'] > 100)).sum()
if n_age_hors > 0:
    print('ATTENTION : {:} contrats avec age hors plage [20,100]'.format(n_age_hors))
